# Day21：LangChain 中间件使用示例

基于本项目的 **LangChain 1.4.3**，覆盖 `langchain.agents.middleware` 导出的全部 **16 种内置中间件**，以及自定义钩子、动态提示词和组合使用。

先运行公共准备单元，再按节运行；各节创建独立 Agent，不依赖前一节的 Agent。通用示例使用本地 Ollama `qwen3.5:9b-q4_K_M`，不需要云端密钥。执行全部单元会产生多次本地推理。

| 类别 | 中间件 |
| --- | --- |
| 上下文 | SummarizationMiddleware、ContextEditingMiddleware |
| 调用控制 | ModelCallLimitMiddleware、ToolCallLimitMiddleware |
| 故障处理 | ModelRetryMiddleware、ToolRetryMiddleware、ModelFallbackMiddleware、ToolErrorMiddleware |
| 输入与人工审核 | PIIMiddleware、HumanInTheLoopMiddleware |
| 工具管理 | TodoListMiddleware、LLMToolSelectorMiddleware、LLMToolEmulator |
| 文件与命令 | FilesystemFileSearchMiddleware、ShellToolMiddleware |
| 厂商功能 | ProviderToolSearchMiddleware |

基本写法：`create_agent(model=model, tools=[...], middleware=[中间件实例, ...])`。

简化执行顺序：`before_agent → [before_model → wrap_model_call(模型) → after_model → wrap_tool_call(工具)]循环 → after_agent`。没有工具调用时跳过工具阶段；`dynamic_prompt` 也在模型调用包装阶段生效。

参考：[内置中间件](https://docs.langchain.com/oss/python/langchain/middleware/built-in)、[自定义中间件](https://docs.langchain.com/oss/python/langchain/middleware/custom)。具体参数以本项目安装版本为准。厂商扩展包中的专用中间件不属于本笔记的覆盖范围。

## 0. 公共准备

复用 Day19/Day20 的本地模型。`show_result` 同时显示实际工具结果和最终回答。

In [1]:
from dataclasses import dataclass
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
import shutil

from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware, AgentState, ModelRequest, ModelResponse, ToolCallRequest,
    SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit,
    ModelCallLimitMiddleware, ToolCallLimitMiddleware,
    ModelRetryMiddleware, ToolRetryMiddleware, ModelFallbackMiddleware,
    ToolErrorMiddleware, PIIMiddleware, HumanInTheLoopMiddleware,
    TodoListMiddleware, LLMToolSelectorMiddleware, LLMToolEmulator,
    FilesystemFileSearchMiddleware, ShellToolMiddleware,
    ProviderToolSearchMiddleware,
    before_agent, before_model, after_model, after_agent,
    wrap_model_call, wrap_tool_call, dynamic_prompt,
)
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langgraph.types import Command

MODEL_OPTIONS = dict(
    model="qwen3.5:9b-q4_K_M",
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    temperature=0,
    reasoning_effort="none",
    max_tokens=1024,
    timeout=120,
    max_retries=0,  # 由后面的中间件示例控制重试。
)
model = ChatOpenAI(**MODEL_OPTIONS)


@tool
def multiply(a: int, b: int) -> int:
    """计算两个整数的乘积。"""
    return a * b


@tool
def get_weather(city: str) -> str:
    """查询城市天气；这里只返回固定教学数据，不访问真实天气服务。"""
    return f"{city}：晴，25°C（固定演示数据）"


def show_result(result):
    for message in result["messages"]:
        if isinstance(message, ToolMessage):
            print(f"工具 {message.name}：{message.content}")
    print("最终回答：", result["messages"][-1].content)


def ask(agent, question):
    result = agent.invoke(
        {"messages": [HumanMessage(question)]},
        config={"recursion_limit": 40},  # 图节点步数，与模型/工具调用次数不同。
    )
    show_result(result)
    return result


print("公共准备完成：", model.model_name)

公共准备完成： qwen3.5:9b-q4_K_M


# @wrap_model_call的使用

In [ ]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call,ModelRequest,ModelResponse

@wrap_model_call
def wrap_model_call_example(request: ModelRequest,handler:Callable[[ModelRequest],ModelResponse]) -> ModelResponse | None:
    request.messages[-1].content+="----->warp_model_call_before<-----"
    # 模型调用
    res=handler(request)
    res.result[0].content+="----->warp_model_call_after<-----"

    return res


## 1. SummarizationMiddleware：历史摘要

历史达到 6 条消息时，把较早消息摘要化，保留最近 2 条。教学阈值刻意设小；实际项目通常按 token 设置。此处直接传入历史，不需要 checkpointer；跨多次调用保存历史时需要它。

In [2]:
summary_agent = create_agent(
    model=model,
    middleware=[SummarizationMiddleware(
        model=model,
        trigger=("messages", 6),
        keep=("messages", 2),
        summary_prompt="用中文简要保留用户的学习目标和偏好：\n{messages}",
    )],
)
history = [
    HumanMessage("我正在学习 LangChain，目标是做企业知识库。"),
    AIMessage("可以先学习模型调用。"),
    HumanMessage("我使用本地 Ollama，喜欢简洁的 Python 示例。"),
    AIMessage("接着学习工具调用和 Agent。"),
    HumanMessage("今天学中间件。"),
    AIMessage("它可以管理上下文和工具调用。"),
    HumanMessage("我的学习目标和代码偏好是什么？简短回答。"),
]
summary_result = summary_agent.invoke({"messages": history})
print("摘要前消息数：", len(history))
print("执行后消息数：", len(summary_result["messages"]))
print("摘要消息：", summary_result["messages"][0].content)
show_result(summary_result)

摘要前消息数： 7
执行后消息数： 4
摘要消息： Here is a summary of the conversation to date:

- **学习目标**：构建企业知识库（基于 LangChain）。
- **当前进度**：已完成模型调用、工具调用与 Agent 的学习，今日重点为中间件。
- **技术偏好**：使用本地 Ollama，偏好简洁的 Python 示例。
最终回答： - **学习目标**：构建企业知识库（基于 LangChain）。
- **代码偏好**：使用本地 Ollama，偏好简洁的 Python 示例。


## 2. ContextEditingMiddleware：清理旧工具结果

超过 token 阈值时用占位符替换旧工具输出，保留最近 1 次结果。与摘要不同，它不额外调用模型，只修改发往模型的请求副本。下面构造合法的历史工具调用，再用观察钩子显示模型真正收到的内容。

In [ ]:
# 把函数转换成“模型调用中间件”，让你在模型调用前后加入自己的逻辑
@wrap_model_call
def inspect_context(request: ModelRequest, handler):
    for message in request.messages:
        if isinstance(message, ToolMessage):
            print("发给模型的工具结果：", str(message.content)[:100])
    return handler(request)


context_agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        ContextEditingMiddleware(edits=[ClearToolUsesEdit(
            trigger=100, keep=1, placeholder="[旧工具结果已清理]",
        )]),
        inspect_context,
    ],
)
context_history = [HumanMessage("先后查询北京和上海天气。")]
for index, city in enumerate(["北京", "上海"]):
    call_id = f"weather_{index}"
    context_history.extend([
        AIMessage(content="", tool_calls=[{
            "name": "get_weather", "args": {"city": city},
            "id": call_id, "type": "tool_call",
        }]),
        ToolMessage(content=f"{city}：晴。" + "历史详细数据。" * 150,
                    name="get_weather", tool_call_id=call_id),
    ])
context_history.append(HumanMessage("只根据最近一次结果，回答最后查询的城市。不要再调用工具。"))
context_result = context_agent.invoke({"messages": context_history})
print("最终回答：", context_result["messages"][-1].content)

发给模型的工具结果： [旧工具结果已清理]
发给模型的工具结果： 上海：晴。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细数据。历史详细
最终回答： 上海：晴。


## 3. ModelCallLimitMiddleware：限制模型调用次数

单次 invoke 最多调用模型 1 次。模型调用工具后，原本需要第二次模型调用生成回答，此时中间件会结束运行并给出限额提示。`thread_limit` 则需要 checkpointer 和相同 thread_id 才能跨 invoke 累计。

In [4]:
model_limit_agent = create_agent(
    model=model, tools=[multiply],
    system_prompt="必须调用 multiply 计算用户给出的乘法，不能心算。",
    middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")],
)
model_limit_result = ask(model_limit_agent, "请调用 multiply 计算 12 × 13。")

工具 multiply：156
最终回答： Model call limits exceeded: run limit (1/1)


## 4. ToolCallLimitMiddleware：限制工具调用次数

只允许 multiply 执行 1 次，超出后结束。用包装器依次模拟两次模型响应，确保稳定触发限额；工具节点真实执行第一次计算。第二次调用被拦截并返回限额消息。若同一批次中存在多个待执行调用，end 会跳过该批次的全部待执行调用。

In [2]:
@wrap_model_call
def supply_limited_calls(request: ModelRequest, handler):
    completed = sum(isinstance(m, ToolMessage) for m in request.messages)
    if completed < 2:
        a, b = [(2, 3), (4, 5)][completed]
        return AIMessage(content="", tool_calls=[{
            "name": "multiply", "args": {"a": a, "b": b},
            "id": f"limit_{completed}", "type": "tool_call",
        }])
    return handler(request)


tool_limit_agent = create_agent(
    model=model, tools=[multiply],
    middleware=[
        ToolCallLimitMiddleware(tool_name="multiply", run_limit=1, exit_behavior="end"),
        supply_limited_calls,
    ],
)
tool_limit_result = ask(tool_limit_agent, "依次计算 2×3 和 4×5。")

工具 multiply：6
工具 multiply：Tool call limit exceeded. Do not call 'multiply' again.
最终回答： 'multiply' tool call limit reached: run limit exceeded (2/1 calls).


## 5. ModelRetryMiddleware：模型失败重试

用自定义包装器模拟第一次调用超时，第二次才真正调用本地模型。列表中重试中间件在外层，能捕获内层抛出的异常。`max_retries=1` 表示首次失败后再试 1 次。

In [3]:
model_attempts = {"count": 0}


@wrap_model_call
def fail_model_once(request: ModelRequest, handler):
    model_attempts["count"] += 1
    print("模型尝试次数：", model_attempts["count"])
    if model_attempts["count"] == 1:
        raise TimeoutError("教学模拟：首次请求超时")
    return handler(request)


retry_agent = create_agent(model=model, middleware=[
    ModelRetryMiddleware(max_retries=1, retry_on=(TimeoutError,),
                         initial_delay=0, jitter=False, on_failure="error"),
    fail_model_once,
])
retry_result = ask(retry_agent, "只回答：重试后模型调用成功。")

模型尝试次数： 1
模型尝试次数： 2
最终回答： 重试后模型调用成功。


## 6. ToolRetryMiddleware：工具失败重试

演示查询工具首次超时、第二次成功。只重试指定工具和指定异常；对有副作用的工具，重试前应保证操作幂等。

In [7]:
tool_attempts = {"count": 0}


@tool
def unstable_lookup(order_id: str) -> str:
    """查询教学订单状态；第一次故意超时，后续返回固定结果。"""
    tool_attempts["count"] += 1
    print("查询尝试次数：", tool_attempts["count"])
    if tool_attempts["count"] == 1:
        raise TimeoutError("教学模拟：查询超时")
    return f"教学订单 {order_id} 已完成"


tool_retry_agent = create_agent(
    model=model, tools=[unstable_lookup],
    system_prompt="必须调用 unstable_lookup 查询订单，成功后直接回答，不重复查询。",
    middleware=[ToolRetryMiddleware(
        tools=["unstable_lookup"], max_retries=1, retry_on=(TimeoutError,),
        initial_delay=0, jitter=False, on_failure="error",
    )],
)
tool_retry_result = ask(tool_retry_agent, "查询订单 DEMO-001。")

查询尝试次数： 1
查询尝试次数： 2
工具 unstable_lookup：教学订单 DEMO-001 已完成
最终回答： 教学订单 DEMO-001 已完成。


## 7. ModelFallbackMiddleware：切换备用模型

主模型故意使用 Ollama 中不存在的名称，真实请求会失败，再回退到已安装的 Qwen3.5。生产环境可以配置多个真实备用模型。

In [8]:
missing_model = ChatOpenAI(**{**MODEL_OPTIONS, "model": "day21-intentionally-missing"})
fallback_agent = create_agent(
    model=missing_model,
    middleware=[ModelFallbackMiddleware(model)],
)
fallback_result = ask(fallback_agent, "只回答：备用模型调用成功。")

最终回答： 备用模型调用成功。


## 8. ToolErrorMiddleware：把可处理的工具异常交回模型

工具执行抛出 ValueError 后，将其转成错误 ToolMessage，让模型解释原因。未匹配的异常继续抛出。它不负责重试，参数校验错误也不经过这个执行异常处理器。

In [9]:
@tool
def divide(a: float, b: float) -> float:
    """计算 a 除以 b，b 不得为零。"""
    if b == 0:
        raise ValueError("除数为零")
    return a / b


def handle_divide_error(exc: Exception, request: ToolCallRequest) -> str | None:
    if isinstance(exc, ValueError):
        return "除数不能为 0，请让用户提供非零除数。"
    return None


@wrap_model_call
def supply_zero_division(request: ModelRequest, handler):
    if not any(isinstance(m, ToolMessage) for m in request.messages):
        return AIMessage(content="", tool_calls=[{
            "name": "divide", "args": {"a": 10, "b": 0},
            "id": "divide_zero", "type": "tool_call",
        }])
    return handler(request)


error_agent = create_agent(
    model=model, tools=[divide],
    system_prompt="工具报错时向用户解释原因，不重复执行同一个失败调用。",
    middleware=[ToolErrorMiddleware(handle_divide_error, tools=["divide"]), supply_zero_division],
)
# 固定首次模型响应，确保执行 divide，而不是被模型提前判断为无效。
error_result = ask(error_agent, "计算 10 除以 0。")

工具 divide：除数不能为 0，请让用户提供非零除数。
最终回答： 无法计算 10 除以 0，因为除数不能为零。数学上，除以零是没有定义的。如果您有其他数字需要计算，请告诉我！


## 9. PIIMiddleware：个人信息处理

将用户输入的演示邮箱替换成占位符。可选策略有 redact、mask、hash、block；还可以通过 detector 正则自定义手机号等类型。`apply_to_output` 用于处理模型输出，不代表阻止模型读取输入。

In [10]:
@wrap_model_call
def inspect_pii_input(request: ModelRequest, handler):
    print("模型实际收到：", request.messages[-1].content)
    return handler(request)


pii_agent = create_agent(model=model, middleware=[
    PIIMiddleware("email", strategy="redact", apply_to_input=True),
    inspect_pii_input,
])
pii_result = ask(pii_agent, "我的演示邮箱是 learner@example.com。用一句话确认收到信息。")
print("状态中的输入：", pii_result["messages"][0].content)

模型实际收到： 我的演示邮箱是 [REDACTED_EMAIL]。用一句话确认收到信息。
最终回答： 已收到您的演示邮箱信息，后续沟通将发送至 [REDACTED_EMAIL]。
状态中的输入： 我的演示邮箱是 [REDACTED_EMAIL]。用一句话确认收到信息。


## 10. HumanInTheLoopMiddleware：暂停、审核、恢复

Agent 请求工具后暂停，取得 action_requests 和 review_configs，再用 Command(resume=...) 恢复同一个 thread。必须配置 checkpointer。本例的工具只返回字符串；approve 是教学用自动构造的审核决定，实际产品应由真实用户提交。

In [11]:
@tool
def prepare_notice(text: str) -> str:
    """生成演示通知文本；不发送消息，不写外部系统。"""
    print("审核通过，工具开始执行。")
    return f"演示通知已准备：{text}"


review_agent = create_agent(
    model=model, tools=[prepare_notice],
    system_prompt="必须调用 prepare_notice 准备通知，成功后直接回答。",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={
        "prepare_notice": {"allowed_decisions": ["approve", "edit", "reject"]},
    })],
    checkpointer=InMemorySaver(),
)
# 每次重跑用新 thread_id，避免读到上一轮已完成的状态。
from uuid import uuid4
review_config = {"configurable": {"thread_id": f"day21-review-{uuid4().hex}"}}
paused = review_agent.invoke(
    {"messages": [HumanMessage("准备通知：明天上午九点学习中间件。请调用工具。") ]},
    config=review_config,
)
review = paused["__interrupt__"][0].value
print("待审核调用：", review["action_requests"])
print("允许的审核决定：", review["review_configs"])
decisions = [{"type": "approve"} for action in review["action_requests"]]
resumed = review_agent.invoke(Command(resume={"decisions": decisions}), config=review_config)
show_result(resumed)
# 拒绝时用 {"type": "reject", "message": "请取消本次操作"}。
# 编辑时用 {"type": "edit", "edited_action": {"name": "prepare_notice", "args": {"text": "新文本"}}}。

待审核调用： [{'name': 'prepare_notice', 'args': {'text': '明天上午九点学习中间件'}, 'description': "Tool execution requires approval\n\nTool: prepare_notice\nArgs: {'text': '明天上午九点学习中间件'}"}]
允许的审核决定： [{'action_name': 'prepare_notice', 'allowed_decisions': ['approve', 'edit', 'reject']}]
审核通过，工具开始执行。
工具 prepare_notice：演示通知已准备：明天上午九点学习中间件
最终回答： 通知已准备：明天上午九点学习中间件。


## 11. TodoListMiddleware：维护任务清单

自动注入 write_todos 工具，工具更新 Agent 状态中的 todos。它管理清单，不会替你实现待办任务。这里仅要求创建计划，因此最后仍有 pending 项是正常的。

In [12]:
todo_agent = create_agent(
    model=model,
    middleware=[TodoListMiddleware()],
    system_prompt="按用户要求调用 write_todos 创建计划，不执行计划中的任务；写入后简短回答。",
)
todo_result = ask(todo_agent,
    "请调用 write_todos，为学习中间件创建三项待办：阅读概念、运行示例、独立修改。"
    "每项 status 都设为 pending。本轮只创建计划。")
print("任务清单：", todo_result.get("todos", []))

工具 write_todos：Updated todo list to [{'content': '阅读中间件概念', 'status': 'pending'}, {'content': '运行中间件示例', 'status': 'pending'}, {'content': '独立修改中间件代码', 'status': 'pending'}]
最终回答： 待办列表已创建，包含三项任务：阅读中间件概念、运行中间件示例、独立修改中间件代码。
任务清单： [{'content': '阅读中间件概念', 'status': 'pending'}, {'content': '运行中间件示例', 'status': 'pending'}, {'content': '独立修改中间件代码', 'status': 'pending'}]


## 12. LLMToolSelectorMiddleware：先筛选工具

先额外调用模型挑选相关工具，再把筛选结果提供给主模型，适合工具较多的 Agent。max_tools 限制数量，不限制执行次数；选择过程本身也消耗推理时间。

In [13]:
@wrap_model_call
def inspect_selected_tools(request: ModelRequest, handler):
    print("主模型可用工具：", [t.name if hasattr(t, "name") else t.get("name") for t in request.tools])
    return handler(request)


selector_agent = create_agent(
    model=model, tools=[multiply, get_weather],
    system_prompt="使用相关工具回答问题，得到结果后直接回答。",
    middleware=[
        LLMToolSelectorMiddleware(model=model, max_tools=1),
        inspect_selected_tools,
    ],
)
selector_result = ask(selector_agent, "调用天气工具查询杭州天气。")

主模型可用工具： ['get_weather']
主模型可用工具： ['get_weather']
工具 get_weather：杭州：晴，25°C（固定演示数据）
最终回答： 杭州天气晴朗，气温25°C。


## 13. LLMToolEmulator：用模型模拟工具输出

指定工具会由 LLM 生成模拟结果，原工具函数不会执行。适合演示和集成调试；模拟结果不是真实业务数据。下面用计数器确认原函数未执行。

In [14]:
emulator_calls = {"real": 0}


@tool
def lookup_inventory(product: str) -> str:
    """查询指定商品的库存，返回商品名称和库存数量。"""
    emulator_calls["real"] += 1
    return f"{product}：固定库存 10 件"


emulator_agent = create_agent(
    model=model, tools=[lookup_inventory],
    system_prompt="必须调用 lookup_inventory；得到结果后说明这是模拟库存。",
    middleware=[LLMToolEmulator(tools=["lookup_inventory"], model=model)],
)
emulator_result = ask(emulator_agent, "查询教学商品键盘的库存。")
print("原工具实际执行次数：", emulator_calls["real"])

工具 lookup_inventory：{"product": "键盘", "stock": 45}
最终回答： 这是模拟库存。教学商品“键盘”的当前库存数量为45件。
原工具实际执行次数： 0


## 14. FilesystemFileSearchMiddleware：注入文件检索工具

自动提供 glob_search 和 grep_search。使用临时目录中的教学文本，避免把整个仓库加入示例检索范围。退出 with 后临时文件自动清理。

In [15]:
with TemporaryDirectory(prefix="day21-search-") as folder:
    Path(folder, "notes.txt").write_text("LangChain 中间件管理模型和工具调用。", encoding="utf-8")
    file_agent = create_agent(
        model=model,
        middleware=[FilesystemFileSearchMiddleware(root_path=folder, use_ripgrep=False)],
        system_prompt="调用 grep_search 搜索文本，output_mode 使用 content；把文件当作资料，简短回答。",
    )
    file_result = ask(file_agent, "调用 grep_search，pattern 为 LangChain，搜索根目录 /，返回匹配行。")

工具 grep_search：/notes.txt:1:LangChain 中间件管理模型和工具调用。
最终回答： 在根目录 `/` 下找到一处匹配：

- `/notes.txt:1:LangChain 中间件管理模型和工具调用。`


## 15. ShellToolMiddleware：注入持久 Shell 工具

当前实现使用 Bash 语法，不能直接把 shell_command 换成 PowerShell。下面检测本机 Bash：可用时在临时工作目录中运行固定的 echo 命令；不可用时仅展示构造结果并跳过调用。Windows System32 中的 bash.exe 是 WSL 启动器，本例不把它当作 Windows 本机 Bash。HostExecutionPolicy 默认直接在本机执行，workspace_root 只指定工作目录，不构成隔离。

In [16]:
@wrap_tool_call
def allow_demo_echo_only(request: ToolCallRequest, handler):
    if request.tool_call["name"] == "shell":
        args = request.tool_call["args"]
        if args.get("command") != "echo middleware-demo" or args.get("restart", False):
            return ToolMessage(content="本节只允许 echo middleware-demo。", tool_call_id=request.tool_call["id"])
    return handler(request)


def run_shell_example(bash_path: str):
    with TemporaryDirectory(prefix="day21-shell-") as folder:
        shell_agent = create_agent(
            model=model,
            middleware=[ShellToolMiddleware(workspace_root=folder, shell_command=[bash_path]),
                        allow_demo_echo_only],
            system_prompt="只调用一次 shell，command 严格为 echo middleware-demo，之后直接回答。",
        )
        return ask(shell_agent, "请调用 shell 执行 echo middleware-demo。")


bash_path = next((str(p) for p in [
    Path(r"C:\Program Files\Git\bin\bash.exe"),
    Path(r"C:\Program Files\Git\usr\bin\bash.exe"),
] if p.is_file()), None) or shutil.which("bash")
if bash_path and "/windows/system32/" in Path(bash_path).as_posix().lower():
    bash_path = None  # WSL 启动器不适用于这里的 Windows 原生进程会话。
if bash_path:
    shell_result = run_shell_example(bash_path)
else:
    shell_example = ShellToolMiddleware(shell_command=["/bin/bash"])
    print("构造成功；当前未找到 Bash，跳过 Shell 执行。可在 Bash 环境调用 run_shell_example(bash_path)。")

构造成功；当前未找到 Bash，跳过 Shell 执行。可在 Bash 环境调用 run_shell_example(bash_path)。


## 16. ProviderToolSearchMiddleware：厂商原生工具搜索

将指定工具标记为延迟加载，并注入厂商服务端工具搜索。它要求提供商和模型支持该接口；Ollama 的 OpenAI 兼容入口不等于支持 OpenAI 服务端工具搜索。下面提供完整调用函数，但不向本地模型发送不受支持的请求，也不创建云端客户端或读取密钥。

In [17]:
def run_provider_search_example(provider_model):
    """传入已配置、支持原生工具搜索的 OpenAI / Anthropic 模型。"""
    provider_agent = create_agent(
        model=provider_model, tools=[multiply, get_weather],
        middleware=[ProviderToolSearchMiddleware(searchable_tools=["get_weather"])],
        system_prompt="需要天气信息时查找并调用 get_weather，再简短回答。",
    )
    return ask(provider_agent, "查找合适的天气工具，查询北京天气。")


provider_search_example = ProviderToolSearchMiddleware(searchable_tools=["get_weather"])
print("构造成功；调用方式：run_provider_search_example(支持工具搜索的模型实例)。本地 Ollama 不执行此节请求。")

构造成功；调用方式：run_provider_search_example(支持工具搜索的模型实例)。本地 Ollama 不执行此节请求。


## 17. 自定义节点钩子：观察 Agent 生命周期

before_agent/after_agent 每次 Agent 运行执行一次；before_model/after_model 在每次模型调用前后执行，因此工具循环时会重复。返回 None 表示只观察状态；需要修改时返回状态更新字典。

In [3]:
@before_agent
def on_agent_start(state: AgentState, runtime: Runtime):
    print("1. before_agent：开始本轮 Agent")


@before_model
def on_model_start(state: AgentState, runtime: Runtime):
    print("2. before_model：当前消息数", len(state["messages"]))


@after_model
def on_model_end(state: AgentState, runtime: Runtime):
    print("3. after_model：本次工具调用数", len(state["messages"][-1].tool_calls))


@after_agent
def on_agent_end(state: AgentState, runtime: Runtime):
    print("4. after_agent：结束本轮 Agent")


hooks_agent = create_agent(
    model=model, tools=[multiply],
    system_prompt="必须调用 multiply 计算，得到结果后直接回答。",
    middleware=[on_agent_start, on_model_start, on_model_end, on_agent_end],
)
hooks_result = ask(hooks_agent, "计算 6 × 7。")

1. before_agent：开始本轮 Agent
2. before_model：当前消息数 1
3. after_model：本次工具调用数 1
2. before_model：当前消息数 3
3. after_model：本次工具调用数 0
4. after_agent：结束本轮 Agent
工具 multiply：42
最终回答： 6 × 7 = 42。


## 18. wrap_model_call：修改请求并统计耗时

handler(request) 才真正执行下一层或模型。通过 request.override 修改本次请求；返回 handler 的响应，不能只返回字符串。

In [19]:
@wrap_model_call
def time_model_call(request: ModelRequest, handler) -> ModelResponse:
    start = perf_counter()
    response = handler(request.override(model_settings={"temperature": 0, "max_tokens": 256}))
    print(f"本次模型调用耗时：{perf_counter() - start:.2f} 秒")
    return response


timed_agent = create_agent(model=model, middleware=[time_model_call])
timed_result = ask(timed_agent, "用一句话解释 Agent。")

本次模型调用耗时：0.49 秒
最终回答： Agent 是一种能够感知环境、自主规划行动并持续学习以完成特定目标的智能体。


## 19. wrap_tool_call：观察和处理工具执行

从 request.tool_call 读取工具名与参数，handler(request) 执行工具。自己构造 ToolMessage 时，tool_call_id 必须匹配原调用的 id。这里直接处理异常的思路也可以用内置 ToolErrorMiddleware 实现。

In [20]:
@wrap_tool_call
def trace_tool(request: ToolCallRequest, handler):
    print("即将执行工具：", request.tool_call["name"], request.tool_call["args"])
    response = handler(request)
    print("工具执行完成，调用 ID：", request.tool_call["id"])
    return response


traced_agent = create_agent(
    model=model, tools=[multiply], middleware=[trace_tool],
    system_prompt="必须调用 multiply 计算，得到结果后直接回答。",
)
traced_result = ask(traced_agent, "计算 8 × 9。")

即将执行工具： multiply {'a': 8, 'b': 9}
工具执行完成，调用 ID： call_n15cjw7q
工具 multiply：72
最终回答： 8 × 9 = 72。


## 20. dynamic_prompt：根据运行上下文生成提示词

通过 context_schema 定义运行上下文，在 invoke(context=...) 传入，再从 request.runtime.context 读取。用户等级属于本次运行上下文，不需要写进聊天历史。

In [21]:
@dataclass
class LearningContext:
    level: str


@dynamic_prompt
def learning_prompt(request: ModelRequest) -> str:
    level = request.runtime.context.level
    if level == "beginner":
        return "你是 Python 老师。面向初学者，用一个生活比喻、两句话解释，不用术语堆砌。"
    return "你是 Python 老师。面向有经验的开发者，用两句话解释技术机制。"


prompt_agent = create_agent(
    model=model, context_schema=LearningContext, middleware=[learning_prompt],
)
prompt_result = prompt_agent.invoke(
    {"messages": [HumanMessage("解释 LangChain 中间件。") ]},
    context=LearningContext(level="beginner"),
)
show_result(prompt_result)

最终回答： LangChain 的中间件就像餐厅里的传菜员，负责在厨师（模型）和顾客（用户）之间高效地传递订单与菜品。它让复杂的对话流程变得井井有条，你只需专注于想做什么，不用操心数据怎么流转。


## 21. AgentMiddleware：类写法与自定义状态

需要在多个钩子间共享 Agent 状态时，可以定义 AgentState 子类并设置 state_schema。计数存在图状态中；使用实例变量保存会造成多个会话共享计数。

In [22]:
class CountingState(AgentState):
    model_rounds: int


class CountRoundsMiddleware(AgentMiddleware):
    state_schema = CountingState

    def before_agent(self, state: CountingState, runtime: Runtime):
        return {"model_rounds": 0}

    def after_model(self, state: CountingState, runtime: Runtime):
        return {"model_rounds": state.get("model_rounds", 0) + 1}

    def after_agent(self, state: CountingState, runtime: Runtime):
        print("本轮模型调用次数：", state["model_rounds"])


count_agent = create_agent(
    model=model, tools=[multiply], middleware=[CountRoundsMiddleware()],
    system_prompt="必须调用 multiply 计算，得到结果后直接回答。",
)
count_result = ask(count_agent, "计算 3 × 9。")
print("返回状态中的 model_rounds：", count_result["model_rounds"])

本轮模型调用次数： 2
工具 multiply：27
最终回答： 3 × 9 = 27。
返回状态中的 model_rounds： 2


## 22. 组合使用：脱敏、限额和工具观测

将中间件实例放在同一个列表中即可组合。before 钩子按列表顺序，after 钩子反向执行；同类 wrap 包装器从左到右嵌套，返回时逐层退出。顺序会影响异常能否被外层捕获和请求能否被内层看到。

In [4]:
@wrap_tool_call
def observe_combined_tool(request: ToolCallRequest, handler):
    print("组合示例执行工具：", request.tool_call["name"], request.tool_call["args"])
    return handler(request)


combined_agent = create_agent(
    model=model, tools=[multiply],
    system_prompt="必须调用 multiply 计算，忽略与计算无关的邮箱，得到结果后简短回答。",
    middleware=[
        PIIMiddleware("email", strategy="redact"),
        ModelCallLimitMiddleware(run_limit=3, exit_behavior="end"),
        ToolCallLimitMiddleware(run_limit=2, exit_behavior="end"),
        observe_combined_tool,
    ],
)
combined_result = ask(combined_agent, "演示邮箱 learner@example.com；请计算 7 × 8。")

组合示例执行工具： multiply {'a': 7, 'b': 8}
工具 multiply：56
最终回答： 7 × 8 = 56。


## 如何确认自己看懂了

1. 把模型调用限额从 1 改成 2，观察工具执行后能否生成最终回答。
2. 把工具重试次数改成 0，观察异常是否继续抛出。
3. 把人工审核的 approve 改成 reject，确认工具体没有执行。
4. 把上下文清理的 keep 改成 2，观察旧结果是否仍保留。
5. 把 LearningContext.level 改成 experienced，比较动态提示词带来的回答变化。

注意区分：中间件已构造、Agent 已编译、调用实际成功、目标分支已触发。这四种状态不等价。Shell 和厂商原生搜索分别受本机 Bash 与服务端能力限制，相关单元会明确显示执行或跳过。

执行验证（2026-10-20）：23 个代码单元均已执行且没有未处理异常，输出保存在各单元下方。通用示例在项目 .venv 中运行；失败重试、回退、调用限额、脱敏及人工审核均触发了对应分支。Shell 仅构造，因缺少适用的本机 Bash 跳过执行；厂商原生工具搜索仅构造，未验证服务端调用。